# Setup & Imports

In [52]:
# ==============================================================================
# 1. Setup & Imports
# ==============================================================================
import yaml
from copy import deepcopy
from pathlib import Path

CONFIG_FILE_NAME = "campaign_config.yaml"


def merge_config_layer(base, override):
    """Merge one config layer into another, appending parameter_sets."""
    result = deepcopy(base)

    for key, value in override.items():
        if key == "parameter_sets":
            existing = result.get(key, []) or []
            additions = deepcopy(value) if value else []
            result[key] = [*existing, *additions]
        elif isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = merge_config_layer(result[key], value)
        else:
            result[key] = deepcopy(value)

    return result


def resolve_simulation_config(campaign_config, sim_id):
    """Merge campaign defaults, mode defaults, and simulation overrides."""
    simulations = campaign_config.get("simulations", [])
    sim_entry = next((entry for entry in simulations if entry.get("id") == sim_id), None)
    if sim_entry is None:
        raise KeyError(f"Simulation '{sim_id}' was not found in the campaign config.")

    merged_config = {}
    defaults = campaign_config.get("simulation_defaults", {})
    mode_defaults = campaign_config.get("mode_defaults", {}).get(sim_entry.get("mode"), {})

    for layer in (defaults, mode_defaults, sim_entry):
        merged_config = merge_config_layer(merged_config, layer)

    return merged_config


# Resolve paths
# __file__ is not available in Jupyter notebooks; fall back to Path().resolve()
# which returns the notebook's working directory (its own folder when run in VS Code).
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path().resolve()

CAMPAIGN_DIR = SCRIPT_DIR
PARAMS_DIR = CAMPAIGN_DIR / "parameters"
OPT_CONFIG_DIR = CAMPAIGN_DIR / "optimization_config"

# ── AnimaKinetica imports ──────────────────────────────────────────────────────
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation_data import SimulationData

from animakinetica.models.kinetics import Kinetics
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController

print("✓ Imports successful")
print(f"  Script dir    : {SCRIPT_DIR}")
print(f"  Parameters dir: {PARAMS_DIR}")
print(f"  Opt config dir: {OPT_CONFIG_DIR}")
print(f"  Config file   : {CONFIG_FILE_NAME}")


✓ Imports successful
  Script dir    : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichiometry_Humbird_enhanced
  Parameters dir: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichiometry_Humbird_enhanced\parameters
  Opt config dir: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichiometry_Humbird_enhanced\optimization_config
  Config file   : campaign_config.yaml


In [53]:
# ==============================================================================
# 2. Load Campaign Config & Parameters
#
# ▶ SET SIM_ID MANUALLY to the simulation you want to troubleshoot.
#   Valid IDs are the "id" fields in campaign_config.yaml (e.g. "batch_E").
# ==============================================================================
SIM_ID = "cp_gsp"   # <── change this to the simulation you want to inspect
print(f"\n Simulation ID: {SIM_ID}")


 Simulation ID: cp_gsp


In [54]:
# ── Load campaign configuration ────────────────────────────────────────────────
config_path = CAMPAIGN_DIR / CONFIG_FILE_NAME
with open(config_path, encoding="utf-8") as f:
    campaign_config = yaml.safe_load(f)

# Resolve campaign defaults, mode defaults, and simulation overrides
sim_config = resolve_simulation_config(campaign_config, SIM_ID)
opt_settings = campaign_config["optimization_settings"]

# ── Simulation settings ────────────────────────────────────────────────────────
# Convert string names from YAML to enum objects using EnumClass["NAME"] lookup.
# Optional strategies (null in YAML → None in Python) are handled with a conditional.
MODE = OperatingMode[sim_config["mode"]]
KINETICS = Kinetics[sim_config["kinetics"]]
STOICHIOMETRY = StoichiometryMode[sim_config["stoichiometry"]] if sim_config.get("stoichiometry") else None
FEED_STRATEGY = FeedStrategy[sim_config["feed_strategy"]] if sim_config["feed_strategy"] else None
PERFUSION_STRATEGY = (
    PerfusionStrategy[sim_config["perfusion_strategy"]]
    if sim_config["perfusion_strategy"]
    else None
)
SOLVER_METHOD = sim_config.get("solver_method", "LSODA")

# ── Parameter file paths ───────────────────────────────────────────────────────
PARAM_FILES = [
    PARAMS_DIR / Path(entry).with_suffix(".yaml")
    for entry in sim_config["parameter_sets"]
]

# ── Optimization settings ──────────────────────────────────────────────────────
if sim_config["optimize"]:
    OBJECTIVES = opt_settings["objectives"]
    WEIGHTS = [opt_settings["objective_weights"][obj] for obj in OBJECTIVES]
    SEED = opt_settings["seed"]
    WORKERS = opt_settings["workers"]
    TIMEOUT = opt_settings.get("timeout", 5.0)
    OPT_CONFIG_FILE = OPT_CONFIG_DIR / f"{sim_config['opt_config']}_optimization.yaml"

# ── Export filename ────────────────────────────────────────────────────────────
EXPORT_FILENAME = opt_settings["filename"].replace("{simulation_id}", SIM_ID)

print(f"✓ Campaign config loaded: {SIM_ID} — {sim_config['label']}")
print(f"  Mode               : {MODE}")
print(f"  Kinetics           : {KINETICS}")
print(f"  Stoichiometry      : {STOICHIOMETRY}")
print(f"  Feed strategy      : {FEED_STRATEGY}")
print(f"  Perfusion strategy : {PERFUSION_STRATEGY}")
print(f"  Solver method      : {SOLVER_METHOD}")
print(f"  Optimize           : {sim_config['optimize']}")
if sim_config["optimize"]:
    print(f"  Opt config         : {OPT_CONFIG_FILE.name}")
    print(f"  Objectives         : {OBJECTIVES}")
    print(f"  Weights            : {WEIGHTS}")
    print(f"  Seed / Workers     : {SEED} / {WORKERS}")
    print(f"  Timeout            : {TIMEOUT}")
print(f"  Export file        : {EXPORT_FILENAME}")

# ── Load parameters ────────────────────────────────────────────────────────────
params = SimulationParameters.load_parameters([str(p) for p in PARAM_FILES])
print("\n✓ Parameters loaded successfully")
print("  Files:")
for p in PARAM_FILES:
    print(f"    {p.relative_to(PARAMS_DIR)}")


✓ Campaign config loaded: cp_gsp — Continuous Perfusion | Humbird Enhanced | Glucose Setpoint Perfusion
  Mode               : Continuous Perfusion
  Kinetics           : Stoichiometric
  Stoichiometry      : StoichiometryMode.CONSTANT
  Feed strategy      : Constant
  Perfusion strategy : Glucose Setpoint
  Solver method      : LSODA
  Optimize           : True
  Opt config         : CP_GSP_optimization.yaml
  Objectives         : ['STY_kg_m3_day', 'medium_use_efficiency_kg_m3', 'nutrient_use_efficiency_kg_kg']
  Weights            : [10.0, 1.0, 1.0]
  Seed / Workers     : 42 / 8
  Timeout            : 10
  Export file        : cp_gsp_opt
Loading parameters from 8 file(s) sequentially...
  [1/8] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichiometry_Humbird_enhanced\parameters\base_parameters.yaml
  [2/8] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichio

In [55]:
# ==============================================================================
# 3. Baseline Simulation
# ==============================================================================

# params["fed_batch.C_Gln_feed"] = 60
# params["fed_batch.C_AA_feed"] = 10


params["perfusion.C_Glc_feed"] = 50
params["perfusion.C_AA_feed"] = 100
params["perfusion.C_Glc_setpoint"] = 40


baseline_sim = CellCultureSimulation(
    parameters=params,
    kinetics=KINETICS,
    stoichiometry_mode=STOICHIOMETRY,
    operating_mode=MODE,
    feed_strategy=FEED_STRATEGY,
    perfusion_strategy=PERFUSION_STRATEGY,
    solver_method=SOLVER_METHOD,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,
)

baseline_result = baseline_sim.run_simulation()

print("\n✓ Baseline simulation completed")
print("\nBaseline Performance Metrics:")
for key, value in baseline_sim.data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


run_simulation execution time: 348.043 ms

✓ Baseline simulation completed

Baseline Performance Metrics:
  total_DCM_harvested_kg: 5271.3354
  STY_kg_m3_day: 10.2956
  medium_use_efficiency_kg_m3: 2.8810
  protein_use_efficiency_kg_kg: 0.1532
  nutrient_use_efficiency_kg_kg: 0.1296
  batch_time_range_h: 0.0000


In [56]:
# Cultivation summary table
baseline_result.get_cultivation_summary_df()

,Batch No.,Termination Reason,Time (h),Setup After Batch (h),Batch Start Elapsed Time (h),Batch End Elapsed Time (h),Accum. Setup Time (h),Accum. Active Time (d),Accum. Elapsed Time (d),Accum. Time (d),...,Accum. Acid Addition Use (L),Accum. Total Inlet Use (L),Harvest Volume (L),Accum. Harvest Volume (L),Final Cell Density (10^6 cells/mL),Final Cell Density (g/L),Cells Harvested (10^12 Cells),Cells Harvested (kg),Accum. Cell Harvest (10^12 Cells),Accum. Biomass Harvest (kg)
0,1,Maximum time reached,960.0,0.0,0.0,960.0,0.0,40.0,40.0,40.0,...,0.0,228715.224404,2735.576222,2735.576222,317.922652,286.130387,732.129923,658.916931,732.129923,658.916931


## Mass Balance Check

In [57]:
baseline_data = baseline_sim.data

print("=" * 70)
print("EXTERNAL BOUNDARY CHON CLOSURE (simulated window)")
print("=" * 70)
boundary_closure = baseline_data.compute_boundary_closure()

rel = {e: v["relative_residual_pct"] for e, v in boundary_closure["elements"].items()}
print(rel)

# Element-level summary
closure_df = baseline_data.build_closure_summary_df(boundary_closure)
print("\nElemental closure summary:")
display(closure_df)

# Max residual
print(f"\nMax absolute residual: {boundary_closure['max_abs_residual']:.6f} mol-atoms")

# Component basis
print("\n" + "-" * 70)
print("Component basis:")
print("-" * 70)
component_df = baseline_data.build_component_specs_df(boundary_closure['component_specs'])
display(component_df)


EXTERNAL BOUNDARY CHON CLOSURE (simulated window)
{'C': -0.00041603485179229026, 'H': -5.808369891536402e-06, 'O': -3.816684659476303e-06, 'N': -0.00037950274121504446}

Elemental closure summary:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,1.916879e+05,1.916871e+05,-0.797488,-0.000416
1,H,2.576239e+07,2.576239e+07,-1.496375,-0.000006
2,O,1.287021e+07,1.287021e+07,-0.491215,-0.000004
3,N,3.344225e+04,3.344212e+04,-0.126914,-0.000380



Max absolute residual: 1.496375 mol-atoms

----------------------------------------------------------------------
Component basis:
----------------------------------------------------------------------


,Component,Side,Description,Amount,Amount Unit,C,H,O,N
0,Fresh glucose in,Input,Total glucose entering with fresh initial char...,1.151199e+04,mol,6.000000,12.000000,6.000000,0.000000
1,Fresh glutamine in,Input,Total glutamine entering with fresh initial ch...,1.206606e+03,mol,5.000000,10.000000,3.000000,2.000000
2,Fresh amino acids in,Input,Total lumped amino-acid pool entering with fre...,2.282393e+04,mol,4.816805,9.679107,2.284731,1.359390
3,Fresh lactate in,Input,Lactate entering from fresh external media only.,0.000000e+00,mol,3.000000,6.000000,3.000000,0.000000
4,Fresh ammonium in,Input,Ammonium entering from fresh external media only.,0.000000e+00,mol,0.000000,3.000000,0.000000,1.000000
5,O2 in,Input,Total oxygen crossing the reactor boundary int...,1.823771e+04,mol,0.000000,0.000000,2.000000,0.000000
6,CO2 in,Input,Total inorganic carbon entering from fresh med...,6.632712e+03,mol,1.000000,0.000000,2.000000,0.000000
7,"Medium, base, and acid water in",Input,Carrier water entering with fresh medium charg...,1.269562e+07,mol,0.000000,2.000000,1.000000,0.000000
8,DCM in,Input,"Dry biomass entering via inoculum, only with f...",1.181297e+01,Cmol,1.000000,1.676912,0.335507,0.205111
9,Glucose out,Output,Total glucose leaving in external liquid outle...,4.417285e+03,mol,6.000000,12.000000,6.000000,0.000000


In [58]:
# ==============================================================================
# CHON Closure Delta Report — Boundary vs Stoichiometric
# ==============================================================================
# Compares the integrated kinetic-rate predictions (stoichiometric closure)
# against the actual reactor boundary fluxes to surface discrepancies.
print("\n" + "=" * 70)
print("CHON CLOSURE DELTA (boundary − stoichiometric)")
print("=" * 70)

try:
    delta_result = baseline_data.compute_closure_delta()

    # 1. Element-level comparison
    delta_summary_df = SimulationData.build_delta_summary_df(
        delta_result['stoich_closure'], delta_result['boundary_closure']
    )
    print("\nElement-level comparison:")
    display(delta_summary_df)

    # 2. Per-component delta
    print("\n" + "-" * 70)
    print("Per-component delta basis:")
    print("-" * 70)
    delta_component_df = SimulationData.build_delta_component_specs_df(delta_result['component_specs'])
    display(delta_component_df)

    print(f"\nMax absolute delta residual: {delta_result['max_abs_residual']:.6f} mol-atoms")

except ValueError as exc:
    print(f"\n⚠ Delta report unavailable: {exc}")
    print("  (Requires performance_mode=False for stoichiometric rate trajectories.)")


CHON CLOSURE DELTA (boundary − stoichiometric)

Element-level comparison:


,Element,Stoich Reactants (mol-atoms),Stoich Products (mol-atoms),Stoich Residual (mol-atoms),Boundary Reactants (mol-atoms),Boundary Products (mol-atoms),Boundary Residual (mol-atoms),Delta Residual (mol-atoms)
0,C,76145.212326,76145.211747,-0.000579,1.916879e+05,1.916871e+05,-0.797488,-0.796909
1,H,152557.885511,152557.884209,-0.001303,2.576239e+07,2.576239e+07,-1.496375,-1.495072
2,O,95626.537059,95626.536625,-0.000434,1.287021e+07,1.287021e+07,-0.491215,-0.490781
3,N,10095.788366,10095.788221,-0.000145,3.344225e+04,3.344212e+04,-0.126914,-0.126769



----------------------------------------------------------------------
Per-component delta basis:
----------------------------------------------------------------------


,Component,Side,Description,Stoich Amount,Boundary Net Amount,Delta Amount,Amount Unit,C,H,O,N
0,Glucose consumed delta,Input,Difference between boundary net flux and integ...,7094.658081,7094.705245,0.047164,mol,6.000000,12.000000,6.000000,0.000000
1,Glutamine consumed delta,Input,Difference between boundary net flux and integ...,1052.241691,1052.184734,-0.056957,mol,5.000000,10.000000,3.000000,2.000000
2,Amino acids consumed delta,Input,Difference between boundary net flux and integ...,5878.597517,5878.773560,0.176043,mol,4.816805,9.679107,2.284731,1.359390
3,O2 consumed delta,Input,Difference between boundary net flux and integ...,18235.425065,18235.425127,0.000063,mol,0.000000,0.000000,2.000000,0.000000
4,Dry biomass delta,Output,Difference between boundary net flux and integ...,48257.418391,48257.392492,-0.025899,Cmol,1.000000,1.676912,0.335507,0.205111
5,Lactate produced delta,Output,Difference between boundary net flux and integ...,1976.739844,1976.779243,0.039399,mol,3.000000,6.000000,3.000000,0.000000
6,Ammonium produced delta,Output,Difference between boundary net flux and integ...,197.673984,197.677924,0.003940,mol,0.000000,3.000000,0.000000,1.000000
7,CO2 produced delta,Output,Difference between boundary net flux and integ...,21957.573823,21957.530782,-0.043040,mol,1.000000,0.000000,2.000000,0.000000
8,H2O produced delta,Output,Difference between boundary net flux and integ...,29590.483389,29590.483633,0.000244,mol,0.000000,2.000000,1.000000,0.000000



Max absolute delta residual: 1.495072 mol-atoms


In [59]:
baseline_sim.data.plot_kinetics()

In [60]:
baseline_sim.data.plot_stoichiometric_yields()

In [61]:
baseline_sim.data.plot_volume_feed(batch_number=1)

In [62]:
fig_list = baseline_sim.data.plot_profiles()
for fig in fig_list:
    fig.show()

## Optimization

In [63]:
# ==============================================================================
# 4. Optimization
#
# Multi-objective optimization using differential evolution. Objectives, weights,
# seed, and workers are all read from campaign_config.yaml → optimization_settings.
# The optimization parameter bounds come from the opt_config file resolved above.
# ==============================================================================
from animakinetica.analysis.optimization import OptimizationManager

# ── Campaign normalization scalars ─────────────────────────────────────────────
# These are the max objective values observed across all baseline simulations and are
# used as a consistent divisor so weights carry identical meaning for every simulation.
NORM_FILE_NAME = "campaign_normalization.yaml"
NORM_FILE = CAMPAIGN_DIR / NORM_FILE_NAME

if NORM_FILE.exists():
    with open(NORM_FILE) as f:
        campaign_norm = yaml.safe_load(f)
    print(f"✓ Loaded campaign normalization from {NORM_FILE.name}")
    for obj, val in campaign_norm.items():
        if isinstance(val, (int, float)):
            print(f"  {obj}: {val:.4g}")
else:
    campaign_norm = None
    print("  ⚠ campaign_normalization.yaml not found; run create_normalization_values.ipynb first.")
    print("    Falling back to per-simulation baseline normalization.")

# ── Initialize OptimizationManager ─────────────────────────────────────────────
opt_manager = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file=str(OPT_CONFIG_FILE),
)

print(f"\n✓ OptimizationManager initialized")
print(f"  Config     : {OPT_CONFIG_FILE.name}")
print(f"  Parameters : {len(opt_manager.opt_config)}")
print("\nParameters to optimize:")
for param, cfg in opt_manager.opt_config.items():
    print(f"  {param}: bounds={cfg['bounds']}")

✓ Loaded campaign normalization from campaign_normalization.yaml
  STY_kg_m3_day: 0.9318
  medium_use_efficiency_kg_m3: 4.043
  nutrient_use_efficiency_kg_kg: 0.4345
Searching specified optimization config: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichiometry_Humbird_enhanced\optimization_config\CP_GSP_optimization.yaml
Loading optimization config from: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_6_fixed_stoichiometry_Humbird_enhanced\optimization_config\CP_GSP_optimization.yaml
Parsing...
Parsed and loaded configuration containing 5 optimization parameters.

✓ OptimizationManager initialized
  Config     : CP_GSP_optimization.yaml
  Parameters : 5

Parameters to optimize:
  perfusion.C_Glc_feed: bounds=(0.01, 100)
  perfusion.C_Gln_feed: bounds=(0.01, 100)
  perfusion.C_AA_feed: bounds=(0.01, 100)
  perfusion.B: bounds=(1e-05, 0.05)
  perfusion.C_Glc_setpoint: bounds=(0.01, 10)


In [64]:
# ── Run optimization ───────────────────────────────────────────────────────────
opt_manager.optimize_parameters(
    objectives=OBJECTIVES,
    weights=WEIGHTS,
    workers=1,
    seed=SEED,
    timeout=TIMEOUT,
    verbose=True,
    normalization_values=campaign_norm,
    debug_rejections=True,
)

print("\n✓ Optimization completed")
print("\nOptimal parameters found:")
for param_name, value in opt_manager.optimal_parameters.items():
    cfg = opt_manager.opt_config[param_name]
    print(f"  {param_name}: {value:.6g} {cfg['unit']}")

Starting optimization with objectives: STY_kg_m3_day (weight: 10.0), medium_use_efficiency_kg_m3 (weight: 1.0), nutrient_use_efficiency_kg_kg (weight: 1.0)
Worker configuration: requested=1, effective=1
Optimizing 5 parameters: ['perfusion.C_Glc_feed', 'perfusion.C_Gln_feed', 'perfusion.C_AA_feed', 'perfusion.B', 'perfusion.C_Glc_setpoint']
Using campaign-level normalization scalars:
  STY_kg_m3_day: 0.9318
  medium_use_efficiency_kg_m3: 4.043
  nutrient_use_efficiency_kg_kg: 0.4345
Running differential evolution...
run_simulation execution time: 415.044 ms
[ObjectiveFunction][reject][mode=CONTINUOUS_PERFUSION] continuous_runtime_incomplete: active_h=1008.6094789946725; target_h=960.0; last=Aa depletion; unique=['Aa depletion']
[ObjectiveFunction][reject][mode=CONTINUOUS_PERFUSION] bounds_check_failed
[ObjectiveFunction][reject][mode=CONTINUOUS_PERFUSION] bounds_check_failed


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# 5. Performance Comparison
# ==============================================================================
comparison_df = opt_manager.compare_performance(verbose=True)
comparison_df


=== PERFORMANCE COMPARISON ===
                         Metric                                                                      Description  Baseline      Unit  Optimized  Improvement
          Medium Use Efficiency               Dry biomass harvested per total medium volume sent to the process.  0.400056     kg/m3   0.420017     0.019962
Nutrient Consumption Efficiency             Dry biomass harvested per net nutrient mass consumed in the reactor.  0.193097     kg/kg   0.193045    -0.000052
        Nutrient Use Efficiency                Dry biomass harvested per total nutrient mass fed to the reactor.  0.041557     kg/kg   0.126013     0.084456
         Protein Use Efficiency                   Biomass protein harvested per proteinaceous nutrient mass fed.  0.032136     kg/kg   0.165289     0.133154
                            STY Space-time yield based on harvested dry biomass, reactor size, and elapsed time.  0.073379 kg/m3/day   0.072711    -0.000668
             batch_time_ra

,Metric,Description,Baseline,Unit,Optimized,Improvement
0,Medium Use Efficiency,Dry biomass harvested per total medium volume ...,0.400056,kg/m3,0.420017,0.019962
1,Nutrient Consumption Efficiency,Dry biomass harvested per net nutrient mass co...,0.193097,kg/kg,0.193045,-0.000052
2,Nutrient Use Efficiency,Dry biomass harvested per total nutrient mass ...,0.041557,kg/kg,0.126013,0.084456
3,Protein Use Efficiency,Biomass protein harvested per proteinaceous nu...,0.032136,kg/kg,0.165289,0.133154
4,STY,Space-time yield based on harvested dry biomas...,0.073379,kg/m3/day,0.072711,-0.000668
5,batch_time_range_h,,9.604990,,10.894539,1.289549


In [ ]:
# ==============================================================================
# 6. Final Simulation Check
# ==============================================================================
final_sim = opt_manager.optimized_simulation

print("\nFinal Performance Metrics:")
for key, value in final_sim.data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


Final Performance Metrics:
  STY_kg_m3_day: 0.0727
  medium_use_efficiency_kg_m3: 0.4200
  protein_use_efficiency_kg_kg: 0.1653
  nutrient_use_efficiency_kg_kg: 0.1260
  nutrient_consumption_efficiency_kg_kg: 0.1930
  batch_time_range_h: 10.8945


In [ ]:
# ── Optimized simulation visualizations ────────────────────────────────────────
final_sim.data.plot_kinetics()

In [ ]:
final_sim.data.plot_stoichiometric_yields()

In [ ]:
final_sim.data.plot_volume_feed()

In [ ]:
fig_list = final_sim.data.plot_profiles()
for fig in fig_list:
    fig.show()

In [ ]:
# ==============================================================================
# 7. Export Results
#
# Three outputs saved to the local export/ directory:
#   c4_{SIM_ID}_opt_{timestamp}.xlsx         — full Excel export
#   c4_{SIM_ID}_opt_{timestamp}.pkl          — complete simulation state (pickle)
#   c4_{SIM_ID}_opt_{timestamp}.parameters.yaml — optimized parameter set
# ==============================================================================
EXPORT_DIR = SCRIPT_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# final_sim
final_sim.save(
    filename=EXPORT_FILENAME,
    file_format="excel",
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=False,
    include_pickle=True,
    include_opt_parameters=True,
    overwrite=True,
    export_dir=EXPORT_DIR,
)

print(f"\n✓ Export complete")
print(f"  Directory : {EXPORT_DIR}")
print("  Files written:")
for f in sorted(EXPORT_DIR.rglob("*")):
    if f.is_file():
        print(f"    {f.relative_to(EXPORT_DIR)}  ({f.stat().st_size / 1024:.1f} KB)")

Simulation data exported to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260707_1746\c4_fedbatch_constant_opt_20260707_1746.xlsx
CellCultureSimulation pickle saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260707_1746\c4_fedbatch_constant_opt_20260707_1746.pkl
Parameters saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260707_1746\c4_fedbatch_constant_opt_20260707_1746_opt_params.yaml

✓ Export complete
  Directory : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export
  Files written:
    c4_fedbatch_constant_opt_20260629_1522\c4_fedbatch